# mapping-sim: features, matches and tracks against the truth

The first layer of an OpenSfM run on a `mapping_sim` render -- detection, pairwise matching,
track building -- scored by construction: every feature's ray is cast through the camera that
rendered it, so each one has the surface point, solid and range it truly saw. A match is then
right or wrong by geometry and by entity, not by whether the reconstruction later converged.
Formats and conventions: `mapping_sim/opensfm_data.py`. Experiments: `mapping-sim-experiments.md`.

In [ ]:
run_dir = "/home/jovyan/datasets/mapping-sim/runs/map-test-20261105-1551"
wrong_px = 10.0  # a match whose true reprojection error exceeds this (full-resolution px) counts as wrong
impure_m = 0.25  # a track whose members' true points sit further than this from their mean counts as impure
debug = False

In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import io, json, pathlib, sys, time, collections
from IPython.display import display as ipy_display, Image as ipy_image

sys.path.insert(0, 'analysis/mapping_sim')  # run from the repo root, as the flight-analysis runner does
import opensfm_data as od
from rig import rig_from_config
from scene import Scene

plt.rcParams.update({'figure.dpi': 110, 'font.size': 9})

def show(fig):
    buf = io.BytesIO()
    fig.savefig(buf, format='png', bbox_inches='tight')
    plt.close(fig)
    ipy_display(ipy_image(data=buf.getvalue()))

RUN = pathlib.Path(run_dir)
OSFM = RUN / 'opensfm'
run = json.loads((RUN / 'run.json').read_text())
RENDER = RUN.parent.parent / 'renders' / run['render']
truth = json.loads((RENDER / 'truth.json').read_text())
cfg = truth['config']
cams = {c.name: c for c in rig_from_config(cfg['rig'])}
if truth['scale'] != 1.0:
    for c in cams.values():
        c.lens = c.lens.scaled(truth['scale'])
scene = Scene.from_config(cfg['scene'])
# What each feature saw, as a code into LABELS: per solid, then 'house star' and 'sky' (solid -1).
solid_names = od.solid_labels(cfg['scene'])
LABELS = ['ground', 'house', 'house star'] + sorted({l for l in solid_names if l.startswith('tree')}, key=lambda s: float(s.split('=')[1]))
LABELS += sorted(set(solid_names) - set(LABELS)) + ['sky']
solid_code = np.array([LABELS.index(l) for l in solid_names] + [LABELS.index('sky')], dtype=np.int8)
STAR = LABELS.index('house star')
def named(codes, name=None):
    return pd.Series(pd.Categorical.from_codes(codes, LABELS), name=name)
names = sorted(truth['shots'])
index = {n: i for i, n in enumerate(names)}
shot = [truth['shots'][n] for n in names]
cam_of = np.array([s['camera'] for s in shot])
capture = np.array([int(n.split('_')[0]) for n in names])
print(f"run {RUN.name}: render {run['render']}, {len(names)} images, cameras {sorted(cams)}, {len(scene.solids)} solids")
print(f"ODM {run['odm_image']} args {run['odm_args']!r}")
print((OSFM / 'profile.log').read_text())

## Assumptions

In [ ]:
# Every rendered image has features and matches, and the tracks file is the version the reader knows.
missing = [n for n in names if not (OSFM / 'features' / f'{n}.features.npz').exists()]
assert not missing, f"no features for {len(missing)} images, e.g. {missing[:3]}"
missing = [n for n in names if not (OSFM / 'matches' / f'{n}_matches.pkl.gz').exists()]
assert not missing, f"no matches for {len(missing)} images, e.g. {missing[:3]}"
assert (OSFM / 'tracks.csv').read_bytes()[:len(od.TRACKS_HEADER)] == od.TRACKS_HEADER

# The pixel convention: a feature's stored colour is the rendered pixel at to_pixels(). If this
# fails, every truth below is cast through the wrong pixel.
from PIL import Image
rng = np.random.default_rng(0)
for n in rng.choice(names, 6, replace=False):
    lens = cams[truth['shots'][n]['camera']].lens
    with np.load(OSFM / 'features' / f'{n}.features.npz') as f:
        uv = od.to_pixels(f['points'], lens.width, lens.height)
        col = f['colors'][:, 0].astype(float)
    img = np.asarray(Image.open(RENDER / 'images' / n).convert('L')).astype(float)
    ij = np.clip(np.floor(uv).astype(int), 0, [lens.width - 1, lens.height - 1])
    r = np.corrcoef(col, img[ij[:, 1], ij[:, 0]])[0, 1]
    print(f"{n}: stored colour vs rendered pixel r = {r:.4f}")
    assert r > 0.98, n

## Detection: what each image's features landed on

In [ ]:
t0 = time.time()
feat_img, feat_uv, feat_scale = [], [], []
truth_cols = collections.defaultdict(list)
offset = np.zeros(len(names) + 1, dtype=np.int64)
for i, n in enumerate(names):
    lens = cams[cam_of[i]].lens
    p = od.read_features(OSFM, n)
    uv = od.to_pixels(p, lens.width, lens.height)
    ft = od.feature_truth(scene, lens, np.array(shot[i]['R_wc']), np.array(shot[i]['center']), uv)
    feat_img.append(np.full(len(p), i, dtype=np.int32))
    feat_uv.append(uv.astype(np.float32))
    feat_scale.append((p[:, 2] * max(lens.width, lens.height)).astype(np.float32))  # px
    for k, v in ft.items():
        truth_cols[k].append(v.astype(np.float32) if v.dtype == np.float64 else v)
    offset[i + 1] = offset[i] + len(p)
F = {'image': np.concatenate(feat_img), 'uv': np.concatenate(feat_uv), 'scale_px': np.concatenate(feat_scale)}
F |= {k: np.concatenate(v) for k, v in truth_cols.items()}
F['lab'] = np.where(F['star'], STAR, solid_code[F['solid']]).astype(np.int8)
del feat_img, feat_uv, feat_scale, truth_cols
per_image = np.diff(offset)
print(f"{len(F['image']):,} features in {time.time() - t0:.0f} s; per image min {per_image.min()} median {np.median(per_image):.0f} max {per_image.max()}")
by_label = named(F['lab']).value_counts(sort=False)
by_label = by_label[by_label > 0]
print((by_label / by_label.sum()).round(4).to_string())

In [ ]:
order = list(by_label.index)
fig, axes = plt.subplots(len(cams), 1, figsize=(12, 2.2 * len(cams)), sharex=True)
fig.suptitle('Features per image by what they landed on', fontweight='bold')
tab = pd.crosstab(F['image'], named(F['lab'])).reindex(columns=order, fill_value=0)
cmap = plt.get_cmap('tab20')
for ax, c in zip(axes, sorted(cams)):
    rows = np.flatnonzero(cam_of == c)
    t = tab.loc[rows]
    ax.stackplot(capture[rows], t.T.values, labels=order, colors=[cmap(k) for k in range(len(order))])
    ax.set_ylabel(c)
    ax.grid(alpha=0.3)
axes[0].legend(fontsize=7, ncol=6, loc='upper right')
axes[-1].set_xlabel('capture')
show(fig)

fig, axes = plt.subplots(1, 2, figsize=(12, 3.5))
for l in order:
    m = F['lab'] == LABELS.index(l)
    if l != 'sky':
        axes[0].hist(F['range'][m], bins=np.linspace(0, 150, 76), histtype='step', label=l)
axes[0].set_xlabel('range (m)'); axes[0].set_ylabel('features'); axes[0].legend(fontsize=7); axes[0].grid(alpha=0.3)
axes[1].hist(F['scale_px'], bins=100, log=True)
axes[1].set_xlabel('feature scale (full-res px)'); axes[1].grid(alpha=0.3)
show(fig)

## Matching: every pair OpenSfM tried, and whether each match is the same surface point

In [ ]:
t0 = time.time()
pair_rows = []
M = collections.defaultdict(list)
for i, n in enumerate(names):
    for other, pairs in od.read_matches(OSFM, n).items():
        j = index[other]
        pair_rows.append((i, j, len(pairs)))
        if len(pairs) == 0:
            continue
        a = offset[i] + pairs[:, 0].astype(np.int64)
        b = offset[j] + pairs[:, 1].astype(np.int64)
        lens_b = cams[cam_of[j]].lens
        proj, front = od.project(lens_b, np.array(shot[j]['R_wc']), np.array(shot[j]['center']), F['point'][a].astype(np.float64))
        err = np.linalg.norm(proj - F['uv'][b], axis=1)
        err[~front | np.isnan(err)] = np.inf  # behind B or A saw sky: nothing B could match
        M['a'].append(a); M['b'].append(b); M['err_px'].append(err.astype(np.float32))
pairs = pd.DataFrame(pair_rows, columns=['i', 'j', 'n'])
pairs['gap'] = np.abs(capture[pairs.i] - capture[pairs.j])
pairs['cams'] = [f'{min(x, y)}-{max(x, y)}' for x, y in zip(cam_of[pairs.i], cam_of[pairs.j])]
M = {k: np.concatenate(v) for k, v in M.items()}
M['sep_m'] = np.linalg.norm(F['point'][M['a']] - F['point'][M['b']], axis=1)
M['same_solid'] = F['solid'][M['a']] == F['solid'][M['b']]
M['wrong'] = (M['err_px'] > wrong_px) | ~M['same_solid']
dup = pairs.assign(k=[tuple(sorted(p)) for p in zip(pairs.i, pairs.j)]).k.duplicated().sum()
print(f"{len(pairs):,} pairs tried ({dup} listed from both sides), {int((pairs.n > 0).sum()):,} kept, {len(M['a']):,} matches in {time.time() - t0:.0f} s")
print(f"wrong (true reprojection > {wrong_px:g} px or a different solid): {M['wrong'].sum():,} = {M['wrong'].mean():.3%}")
print(f"true reprojection error, px: p50 {np.median(M['err_px']):.2f}  p90 {np.percentile(M['err_px'], 90):.2f}  p99 {np.percentile(M['err_px'], 99):.2f}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 3.6))
e = np.clip(M['err_px'], 1e-3, 1e4)
axes[0].hist(e, bins=np.logspace(-2, 4, 121), log=True)
axes[0].axvline(wrong_px, color='crimson', ls='--', label=f'wrong_px {wrong_px:g}')
axes[0].set_xscale('log'); axes[0].set_xlabel('true reprojection error in B (full-res px)'); axes[0].legend(); axes[0].grid(alpha=0.3)

g = pairs.groupby('gap').agg(tried=('n', 'size'), kept=('n', lambda x: (x > 0).sum()), matches=('n', 'sum'))
axes[1].bar(g.index, g.tried, color='lightgray', label='tried')
axes[1].bar(g.index, g.kept, color='steelblue', label='kept')
axes[1].set_xlabel('capture gap'); axes[1].set_ylabel('pairs'); axes[1].set_xlim(-1, min(g.index.max(), 300) + 1); axes[1].legend(); axes[1].grid(alpha=0.3)

mg = pd.DataFrame({'gap': np.abs(capture[F['image'][M['a']]] - capture[F['image'][M['b']]]), 'wrong': M['wrong']})
w = mg.groupby('gap').wrong.agg(['mean', 'size'])
axes[2].scatter(w.index, w['mean'], s=np.clip(w['size'] / w['size'].max() * 60, 2, 60))
axes[2].set_xlabel('capture gap'); axes[2].set_ylabel('fraction wrong'); axes[2].set_xlim(-1, min(w.index.max(), 300) + 1); axes[2].grid(alpha=0.3)
show(fig)

print('matches and fraction wrong, by what A saw:')
print(pd.DataFrame({'label': named(F['lab'][M['a']]), 'wrong': M['wrong']}).groupby('label', observed=True).wrong.agg(matches='size', wrong='mean').round(4).to_string())
print()
print('by camera pair:')
print(pairs[pairs.n > 0].groupby('cams').n.agg(pairs='size', matches='sum', median='median').to_string())

In [ ]:
# Wrong matches by what each side truly saw: the off-diagonal is two unrelated surfaces matched.
w = M['wrong']
conf = pd.crosstab(named(F['lab'][M['a'][w]], 'A saw'), named(F['lab'][M['b'][w]], 'B saw'))
print(conf.to_string())

## Tracks: chain length, purity, and how far each one reaches in time

In [ ]:
t0 = time.time()
track_names, T = od.read_tracks(OSFM / 'tracks.csv')
remap = np.array([index[n] for n in track_names])
T['image'] = remap[T['image']]
T['g'] = offset[T['image']] + T['feature']
srt = np.argsort(T['track'], kind='stable')
T = {k: v[srt] for k, v in T.items()}
starts = np.flatnonzero(np.r_[True, T['track'][1:] != T['track'][:-1]])
length = np.diff(np.r_[starts, len(T['track'])])
cap = capture[T['image']]
span = np.maximum.reduceat(cap, starts) - np.minimum.reduceat(cap, starts)
pts = F['point'][T['g']].astype(np.float64)
sky = np.isnan(pts[:, 0])
mean = np.add.reduceat(np.where(sky[:, None], 0, pts), starts) / np.maximum(np.add.reduceat((~sky).astype(int), starts), 1)[:, None]
dist = np.linalg.norm(pts - np.repeat(mean, length, axis=0), axis=1)
spread = np.fmax.reduceat(np.where(sky, np.nan, dist), starts)
sol = F['solid'][T['g']]
solids_per_track = np.array([len(set(x)) for x in np.split(sol, starts[1:])])
caps_per_track = np.array([len(set(x)) for x in np.split(cap, starts[1:])])
tracks = pd.DataFrame({'length': length, 'captures': caps_per_track, 'span': span, 'spread_m': spread, 'solids': solids_per_track,
                       'label': named(F['lab'][T['g'][starts]])})
tracks['impure'] = (tracks.solids > 1) | (tracks.spread_m > impure_m)
print(f"{len(tracks):,} tracks, {len(T['track']):,} observations, read in {time.time() - t0:.0f} s")
print(tracks[['length', 'captures', 'span', 'spread_m']].describe(percentiles=[.1, .5, .9, .99]).round(3).to_string())
print(f"tracks touching more than one solid: {(tracks.solids > 1).mean():.3%};  spread over {impure_m:g} m: {(tracks.spread_m > impure_m).mean():.3%}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 3.6))
L = np.bincount(tracks.length)
axes[0].bar(np.arange(len(L)), L, log=True); axes[0].set_xlabel('track length (observations)'); axes[0].set_xlim(0, min(len(L), 80)); axes[0].grid(alpha=0.3)
S = np.bincount(tracks.span)
axes[1].bar(np.arange(len(S)), S, log=True); axes[1].set_xlabel('capture span (last - first)'); axes[1].grid(alpha=0.3)
im = tracks.groupby(np.minimum(tracks.length, 40)).impure.mean()
axes[2].plot(im.index, im.values, 'o-'); axes[2].set_xlabel('track length (40 = 40+)'); axes[2].set_ylabel('fraction impure'); axes[2].grid(alpha=0.3)
show(fig)

## Starvation: how much of each image's feature budget ends up in a track, and where

In [ ]:
in_track = np.zeros(len(F['image']), dtype=bool)
in_track[T['g']] = True
# Per capture, features in tracks that reach another capture, and one at least 5 captures away.
tr_of_obs = np.repeat(np.arange(len(tracks)), length)
cmin = np.repeat(np.minimum.reduceat(cap, starts), length)
cmax = np.repeat(np.maximum.reduceat(cap, starts), length)
far = np.maximum(cap - cmin, cmax - cap)
img_df = pd.DataFrame({'image': np.arange(len(names)), 'camera': cam_of, 'capture': capture, 'features': per_image,
                       'tracked': np.bincount(F['image'][in_track], minlength=len(names)),
                       'reach_1': np.bincount(T['image'][far >= 1], minlength=len(names)),
                       'reach_5': np.bincount(T['image'][far >= 5], minlength=len(names))})
fig, axes = plt.subplots(len(cams), 1, figsize=(12, 2.2 * len(cams)), sharex=True, sharey=True)
fig.suptitle('Per image: features, those in a track, reaching another capture, reaching 5+ captures away', fontweight='bold')
for ax, c in zip(axes, sorted(cams)):
    d = img_df[img_df.camera == c]
    for col in ['features', 'tracked', 'reach_1', 'reach_5']:
        ax.plot(d.capture, d[col], lw=0.8, label=col)
    ax.set_ylabel(c); ax.grid(alpha=0.3)
axes[0].legend(fontsize=7, ncol=4)
axes[-1].set_xlabel('capture')
show(fig)
print(img_df[['features', 'tracked', 'reach_1', 'reach_5']].describe(percentiles=[.01, .1, .5]).round(0).to_string())

fig, axes = plt.subplots(2, len(cams), figsize=(3.4 * len(cams), 4.2))
for k, c in enumerate(sorted(cams)):
    lens = cams[c].lens
    m = cam_of[F['image']] == c
    for r, (sel, title) in enumerate([(m, 'all features'), (m & in_track, 'in a track')]):
        axes[r, k].hist2d(F['uv'][sel, 0], F['uv'][sel, 1], bins=[48, 27], range=[[0, lens.width], [0, lens.height]], cmap='magma')
        axes[r, k].invert_yaxis(); axes[r, k].set_xticks([]); axes[r, k].set_yticks([])
        axes[r, k].set_title(f'{c}: {title}', fontsize=8)
show(fig)

## Tracks as scene points: how many, how dense, and the geometry of their sightings

Each track is taken as one point in the scene: the mean of its observations' true surface points
(`mean` above; observations of sky are left out). Per track:

- **time in view**: from the first to the last capture time (`time_s` in `truth.json`) of the
  images that saw it;
- **sighting rays**: unit vectors from the point to the optical centre of each image that saw it.
  The **angle subtended** is the largest angle between any two of them, and **range** is the
  median distance to those centres.

Density is counted two ways: ground points per 1 m cell seen from above, over the area within 40 m
of the path, and points on every other kind of solid per m² of its exposed walls and top. The rays
figure takes a random sample of tracks with at least five sightings within 25 m of the middle of
the path, and draws a short line from each point toward every centre that saw it.

In [ ]:
t0 = time.time()
centre_of = np.array([s['center'] for s in shot])
time_of = np.array([s['time_s'] for s in shot])
tm = time_of[T['image']]
tracks['in_view_s'] = np.maximum.reduceat(tm, starts) - np.minimum.reduceat(tm, starts)
# Angle subtended and median range, per track: tracks of one length at a time, so nothing is padded,
# in chunks that keep the pairwise cosines to about 2e7 values.
max_angle = np.full(len(tracks), np.nan)
range_m = np.full(len(tracks), np.nan)
for L in np.unique(length):
    of_len = np.flatnonzero(length == L)
    for js in np.array_split(of_len, len(of_len) * L * L // 20_000_000 + 1):
        idx = starts[js, None] + np.arange(L)
        v = centre_of[T['image'][idx]] - mean[js, None]
        dist_c = np.linalg.norm(v, axis=2)
        u = v / dist_c[..., None]
        cos = np.einsum('bik,bjk->bij', u, u).min(axis=(1, 2))
        max_angle[js] = np.degrees(np.arccos(np.clip(cos, -1, 1)))
        range_m[js] = np.median(dist_c, axis=1)
kind = tracks.label.astype(str).str.split().str[0]  # 'house star' counts as house
seen = (kind != 'sky').to_numpy()  # a track of sky alone has no point
tracks['max_angle_deg'] = np.where(seen, max_angle, np.nan)
tracks['range_m'] = np.where(seen, range_m, np.nan)
tracks['kind'] = kind
print(f"geometry for {len(tracks):,} tracks in {time.time() - t0:.0f} s")
print(tracks.groupby('kind').agg(tracks=('length', 'size'), sightings_median=('length', 'median'),
                                 in_view_s_median=('in_view_s', 'median'), in_view_s_p90=('in_view_s', lambda x: x.quantile(0.9)),
                                 angle_deg_median=('max_angle_deg', 'median'), range_m_median=('range_m', 'median')).round(2).to_string())
print('time in view, s:', ' '.join(f"p{q}={tracks.in_view_s.quantile(q / 100):.0f}" for q in (50, 75, 90, 99)),
      f" max {tracks.in_view_s.max():.0f};  0 s: {(tracks.in_view_s == 0).mean():.1%}  >=10 s: {(tracks.in_view_s >= 10).mean():.1%}  >=60 s: {(tracks.in_view_s >= 60).mean():.1%}")

# Density: ground per 1 m cell from above; every other kind per m² of exposed walls and top.
path = centre_of[np.lexsort((cam_of, time_of))][::len(cams)]  # one centre per capture, in time order
lo_xy, hi_xy = path[:, :2].min(axis=0) - 40, path[:, :2].max(axis=0) + 40
cells = [np.arange(lo_xy[0], hi_xy[0] + 1), np.arange(lo_xy[1], hi_xy[1] + 1)]
gnd = (kind == 'ground').to_numpy()
H, xe, ye = np.histogram2d(mean[gnd, 0], mean[gnd, 1], bins=cells)
occupied = H[H > 0]
print(f"ground points per m² over occupied 1 m cells: p10 {np.percentile(occupied, 10):.1f}  median {np.median(occupied):.1f}  p90 {np.percentile(occupied, 90):.1f}")
area = collections.Counter()
for name, s in zip(solid_names, scene.solids):
    if s.kind == 'box':
        sx, sy, sz = s.size
        area[name.split()[0]] += 2 * (sx + sy) * sz + sx * sy
    elif s.kind == 'cylinder':
        area[name.split()[0]] += 2 * np.pi * s.radius * (s.z1 - s.z0) + np.pi * s.radius ** 2
for k, a in sorted(area.items()):
    print(f"{k}: {(kind == k).sum() / a:.1f} points per m² over {a:,.0f} m²")

In [ ]:
from matplotlib.colors import LogNorm

fig, axes = plt.subplots(1, 2, figsize=(14, 6.4), constrained_layout=True)
im = axes[0].pcolormesh(xe, ye, H.T, cmap='Blues', norm=LogNorm(vmin=1, vmax=max(H.max(), 2)))
fig.colorbar(im, ax=axes[0], label='ground points per m²')
axes[0].set_title(f'Ground track points ({gnd.sum():,})')
in_view_sum = np.histogram2d(mean[gnd, 0], mean[gnd, 1], bins=cells, weights=tracks.in_view_s.to_numpy()[gnd])[0]
with np.errstate(invalid='ignore', divide='ignore'):
    in_view_mean = np.where(H >= 5, in_view_sum / H, np.nan)  # cells with fewer than 5 points left blank
im = axes[1].pcolormesh(xe, ye, np.ma.masked_invalid(in_view_mean).T, cmap='Purples', vmin=0, vmax=np.nanpercentile(in_view_mean, 99))
fig.colorbar(im, ax=axes[1], label='mean time in view (s)')
axes[1].set_title('Time in view of ground points, mean per cell of 5+')
for ax in axes:
    ax.plot(path[:, 0], path[:, 1], color='#444', lw=0.8)
    ax.set_aspect('equal'); ax.set_xlabel('east (m)'); ax.set_ylabel('north (m)')
show(fig)

fig, ax = plt.subplots(figsize=(8, 3.6))
for k in sorted(set(kind) - {'sky'}):
    m = (kind == k).to_numpy()
    ax.hist(tracks.in_view_s[m], bins=np.arange(0, tracks.in_view_s.max() + 2), histtype='step', density=True, label=f'{k} ({m.sum():,})')
ax.set_yscale('log'); ax.set_xlabel('time in view (s)'); ax.set_ylabel('share of tracks per second'); ax.legend(); ax.grid(alpha=0.3)
show(fig)

# Sighting rays: a short line from each sampled point toward every centre that saw it.
mid = np.median(path[:, :2], axis=0)
near = seen & (np.abs(mean[:, 0] - mid[0]) < 25) & (np.abs(mean[:, 1] - mid[1]) < 25) & (length >= 5)
sample = np.random.default_rng(0).choice(np.flatnonzero(near), min(300, near.sum()), replace=False)
colour = {k: plt.get_cmap('tab10')(i) for i, k in enumerate(sorted(set(kind) - {'sky'}))}
fig = plt.figure(figsize=(15, 7), constrained_layout=True)
ax2 = fig.add_subplot(1, 2, 1)
ax3 = fig.add_subplot(1, 2, 2, projection='3d')
for j in sample:
    p = mean[j]
    v = centre_of[T['image'][starts[j]:starts[j] + length[j]]] - p
    ends = p + 3.0 * v / np.linalg.norm(v, axis=1, keepdims=True)
    for e_ in ends:
        ax2.plot([p[0], e_[0]], [p[1], e_[1]], color=colour[kind[j]], lw=0.4, alpha=0.6)
        ax3.plot([p[0], e_[0]], [p[1], e_[1]], [p[2], e_[2]], color=colour[kind[j]], lw=0.4, alpha=0.6)
ax2.plot(path[:, 0], path[:, 1], color='#444', lw=0.8)
ax2.set_xlim(mid[0] - 40, mid[0] + 40); ax2.set_ylim(mid[1] - 40, mid[1] + 40)
ax2.set_aspect('equal'); ax2.set_xlabel('east (m)'); ax2.set_ylabel('north (m)')
ax2.set_title(f'{len(sample)} tracks of 5+ sightings within 25 m of the middle of the path:\n3 m toward each centre that saw it; path in grey')
for k, c in colour.items():
    ax2.plot([], [], color=c, label=k)
ax2.legend(fontsize=8)
ax3.set_xlim(mid[0] - 26, mid[0] + 26); ax3.set_ylim(mid[1] - 26, mid[1] + 26); ax3.set_zlim(0, mean[sample, 2].max() + 3)
ax3.view_init(elev=25, azim=-60); ax3.set_title('the same, oblique')
show(fig)

## Position error of reconstructed points against the geometry of their sightings

The largest reconstruction in `reconstruction.json` is brought into the scene frame by the
similarity transform `score.py` fits to the camera centres. OpenSfM keys each reconstructed point
by its track id, so a point's error is the distance from its fitted position to that track's true
point. Impure tracks are left out, since they have no single true point.

Three views:

- the best and the worst 1% of points by error, compared on time in view, angle subtended, range
  and number of sightings;
- each property's rank correlation (Spearman) with error, over all points and then within cells
  of equal deciles of one or two other properties. Holding the others fixed separates properties
  that can move together, such as time in view, angle subtended and range. Range over the angle in
  radians is included as the scale a triangulation error would take from geometry alone;
- median error, and beside it median range, over a grid of angle-subtended deciles by time in view.

In [ ]:
from scipy.stats import spearmanr
from score import rodrigues, umeyama

t0 = time.time()
rec = max(json.loads((OSFM / 'reconstruction.json').read_text()), key=lambda r: len(r['shots']))
rec_names = [n for n in rec['shots'] if n in index]
C_rec = np.array([-rodrigues(np.array(rec['shots'][n]['rotation'])).T @ np.array(rec['shots'][n]['translation']) for n in rec_names])
s_fit, R_fit, t_fit = umeyama(C_rec, centre_of[[index[n] for n in rec_names]])
point_ids = np.array([int(k) for k in rec['points']])
fitted = (s_fit * (R_fit @ np.array([p['coordinates'] for p in rec['points'].values()]).T)).T + t_fit
del rec
track_ids = T['track'][starts]
row = np.searchsorted(track_ids, point_ids)
assert (track_ids[np.minimum(row, len(track_ids) - 1)] == point_ids).all(), 'a reconstructed point with no track'
pts_df = tracks.iloc[row][['kind', 'length', 'in_view_s', 'max_angle_deg', 'range_m', 'impure']].reset_index(drop=True)
pts_df['err_m'] = np.linalg.norm(fitted - mean[row], axis=1)
pts_df['range_over_angle'] = pts_df.range_m / np.radians(np.maximum(pts_df.max_angle_deg, 1e-3))
print(f"{len(pts_df):,} reconstructed points in {time.time() - t0:.0f} s; impure {pts_df.impure.sum():,}")
pure_pts = pts_df[~pts_df.impure]
q = lambda x: '  '.join(f"p{p}={np.percentile(x, p):.3f}" for p in (50, 90, 99))  # noqa: E731
print(f"error, pure points (m): {q(pure_pts.err_m)}  max {pure_pts.err_m.max():.1f}")
print(f"error, impure points (m): {q(pts_df.err_m[pts_df.impure])}")

props = ['in_view_s', 'max_angle_deg', 'range_m', 'length']
lo_e, hi_e = pure_pts.err_m.quantile([0.01, 0.99])
best, worst = pure_pts[pure_pts.err_m <= lo_e], pure_pts[pure_pts.err_m >= hi_e]
print(f"\nmedians: best 1% (error <= {lo_e:.4f} m), worst 1% (>= {hi_e:.3f} m), all pure points")
print(pd.DataFrame({'best 1%': best[props].median(), 'worst 1%': worst[props].median(), 'all': pure_pts[props].median()}).round(2).to_string())
print(pd.DataFrame({'best 1%': best.kind.value_counts(normalize=True), 'worst 1%': worst.kind.value_counts(normalize=True),
                    'all': pure_pts.kind.value_counts(normalize=True)}).round(3).fillna(0).to_string())

rho = {p: spearmanr(pure_pts[p], pure_pts.err_m)[0] for p in props + ['range_over_angle']}
print('\nSpearman rho with error, all pure points:')
print(pd.Series(rho).round(3).to_string())
within = {}
print('\nSpearman rho with error within deciles of the others (median over cells of 200+ points; p10 to p90 across them):')
for a, held in [('in_view_s', ['max_angle_deg']), ('max_angle_deg', ['in_view_s']), ('max_angle_deg', ['range_m']),
                ('range_m', ['max_angle_deg']), ('in_view_s', ['max_angle_deg', 'range_m'])]:
    cell = sum(pd.qcut(pure_pts[b], 10, labels=False, duplicates='drop') * 10 ** k for k, b in enumerate(held))
    rs = [spearmanr(g[a], g.err_m)[0] for _, g in pure_pts.groupby(cell) if len(g) >= 200 and g[a].nunique() > 1]
    within[f"{a} | {' x '.join(held)}"] = float(np.nanmedian(rs))
    print(f"  {a:14s} within {' x '.join(held):26s} {np.nanmedian(rs):+.3f}   ({np.nanpercentile(rs, 10):+.2f} to {np.nanpercentile(rs, 90):+.2f}, {len(rs)} cells)")

# Median error, and median range, over a grid of angle subtended and time in view.
a_edges = np.r_[0, np.percentile(pure_pts.max_angle_deg, np.arange(10, 100, 10)), 180]
t_edges = np.r_[-0.5, 0.5, 1.5, 3.5, 7.5, 15.5, 31.5, 63.5, np.inf]
grid = pure_pts.groupby([pd.cut(pure_pts.max_angle_deg, a_edges), pd.cut(pure_pts.in_view_s, t_edges)], observed=False)
enough = grid.size().unstack() >= 50
fig, axes = plt.subplots(1, 2, figsize=(15, 4.8), constrained_layout=True)
for ax, col, cmap, label in [(axes[0], 'err_m', 'Oranges', 'median error (m)'), (axes[1], 'range_m', 'Greys', 'median range (m)')]:
    med = grid[col].median().unstack().where(enough)
    im = ax.imshow(med.to_numpy(), origin='lower', aspect='auto', cmap=cmap, norm=LogNorm())
    ax.set_yticks(range(len(med.index)), [f'{i.left:.1f}-{i.right:.1f}' for i in med.index])
    ax.set_xticks(range(len(med.columns)), ['0', '1', '2-3', '4-7', '8-15', '16-31', '32-63', '64+'])
    ax.set_ylabel('angle subtended (deg), deciles'); ax.set_xlabel('time in view (s)')
    fig.colorbar(im, ax=ax, label=f'{label}, cells of 50+ points')
axes[0].set_title('Median position error of pure points by angle subtended and time in view')
axes[1].set_title('Median range of the same points')
show(fig)

## Summary

In [ ]:
summary = {
    'run': RUN.name, 'render': run['render'], 'odm_image': run['odm_image'], 'odm_args': run['odm_args'],
    'wrong_px': wrong_px, 'impure_m': impure_m,
    'images': len(names),
    'features': {'total': int(len(F['image'])), 'per_image_median': float(np.median(per_image)),
                 'by_label': {k: int(v) for k, v in by_label.items()}},
    'pairs': {'tried': int(len(pairs)), 'kept': int((pairs.n > 0).sum())},
    'matches': {'total': int(len(M['a'])), 'wrong': int(M['wrong'].sum()),
                'reprojection_px': {q: float(np.percentile(M['err_px'], int(q[1:]))) for q in ['p50', 'p90', 'p99']}},
    'tracks': {'count': int(len(tracks)), 'observations': int(len(T['track'])),
               'length_median': float(tracks.length.median()), 'span_median': float(tracks.span.median()),
               'span_p90': float(tracks.span.quantile(0.9)), 'single_capture': int((tracks.captures == 1).sum()),
               'impure': int(tracks.impure.sum())},
    'per_image_median': {k: float(img_df[k].median()) for k in ['tracked', 'reach_1', 'reach_5']},
    'track_points': {'by_kind': {k: int(v) for k, v in kind.value_counts().items()},
                     'in_view_s': {f'p{p}': float(tracks.in_view_s.quantile(p / 100)) for p in (50, 75, 90, 99)},
                     'max_angle_deg_median': float(tracks.max_angle_deg.median()),
                     'ground_per_m2_occupied_median': float(np.median(occupied)),
                     'per_m2_of_surface': {k: float((kind == k).sum() / a) for k, a in area.items()}},
    'point_error': {'reconstructed': int(len(pts_df)), 'pure': int(len(pure_pts)),
                    'err_m': {f'p{p}': float(np.percentile(pure_pts.err_m, p)) for p in (50, 90, 99)},
                    'spearman': {k: float(v) for k, v in rho.items()}, 'spearman_within_deciles': within},
}
out = RUN / 'derived'
out.mkdir(exist_ok=True)
(out / 'mapping-sim-features.json').write_text(json.dumps(summary, indent=1))
if debug:
    np.savez_compressed(out / 'mapping-sim-features-matches.npz', **{k: v for k, v in M.items() if k != 'label'})
print(json.dumps(summary, indent=1))